[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CharlesShang/TorchCode/blob/master/solutions/57_rope_scaling_solution.ipynb)

# 🟡 Solution: RoPE Scaling

Reference solution for `rope_scaling`.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch


In [ ]:
# ✅ SOLUTION

def _rotate_half_scaled(x: torch.Tensor) -> torch.Tensor:
    x1 = x[..., 0::2]
    x2 = x[..., 1::2]
    return torch.stack((-x2, x1), dim=-1).flatten(-2)


def apply_scaled_rope(x: torch.Tensor, positions: torch.Tensor | None = None,
                      base: float = 10000.0, scaling_factor: float = 1.0) -> torch.Tensor:
    D = x.shape[-1]
    if D % 2 != 0:
        raise ValueError("last dimension must be even")
    S = x.shape[-2]
    if positions is None:
        positions = torch.arange(S, device=x.device, dtype=x.dtype)
    else:
        positions = positions.to(device=x.device, dtype=x.dtype)
    positions = positions / scaling_factor
    inv_freq = 1.0 / (base ** (torch.arange(0, D, 2, device=x.device, dtype=x.dtype) / D))
    angles = positions.unsqueeze(-1) * inv_freq
    cos = torch.repeat_interleave(torch.cos(angles), 2, dim=-1)
    sin = torch.repeat_interleave(torch.sin(angles), 2, dim=-1)
    while cos.ndim < x.ndim:
        cos = cos.unsqueeze(0)
        sin = sin.unsqueeze(0)
    return x * cos + _rotate_half_scaled(x) * sin


In [ ]:
# Verify
x = torch.randn(2, 4, 8)
y = apply_scaled_rope(x, scaling_factor=2.0)
print(y.shape)
print(torch.allclose(y.norm(dim=-1), x.norm(dim=-1), atol=1e-5))


In [ ]:
# Run judge
from torch_judge import check
check('rope_scaling')
